# Imports


In [1]:
import numpy as np
from pathlib import Path
import collections
from typing import Callable, Dict, Optional, Sequence, Tuple, Union
import importlib

import torch
import torch.nn as nn
import math

import torchvision
from rich import print

from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from tqdm.auto import tqdm

/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import sys

src_parent = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(src_parent)

import scripts
from scripts.image_dataset import PushTImageDataset, normalize_data, unnormalize_data
from scripts.pusht_image_env import PushTImageEnv

from scripts.models import VisionEncoder, ConditionalUnet1D
from scripts.training_Inference import UnetTrainer

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/home/thankgod/2025/msc_project/.venv/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [3]:
from scripts.model_config import TrainingConfig
from skvideo.io import vwrite
from IPython.display import Video

# Environment Setup


In [4]:
# 0. create env object
env = PushTImageEnv()

# 1. seed env for initial state.
# Seed 0-200 are used for the demonstration dataset.
env.seed(1000)

# 2. must reset before use
obs, info = env.reset()

# 3. 2D positional action space [0,512]
action = env.action_space.sample()

# 4. Standard gym step method
obs, reward, terminated, truncated, info = env.step(action)

# prints and explains each dimension of the observation and action vectors
with np.printoptions(precision=4, suppress=True, threshold=5):
    print("obs['image'].shape:", obs['image'].shape, "float32, [0,1]")
    print("obs['agent_pos'].shape:", obs['agent_pos'].shape, "float32, [0,512]")
    print("action.shape: ", action.shape, "float32, [0,512]")

obs['image'].shape:
(3, 96, 96)
float32, [0,1]

obs['agent_pos'].shape:
(2,)
float32, [0,512]

action.shape: 
(2,)
float32, [0,512]

# DataSet


In [5]:
dataset_path = Path.cwd().parent / "data/low_dim_pusht" / "replay_buffer.zarr"

# parameters
pred_horizon = 16
obs_horizon = 2
action_horizon = 8
#|o|o|                             observations: 2
#| |a|a|a|a|a|a|a|a|               actions executed: 8
#|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p| actions predicted: 16

# create dataset from file
dataset = PushTImageDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)
# save training data statistics (min, max) for each dim
stats = dataset.stats

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,
    num_workers=4,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

# visualize data in batch
batch = next(iter(dataloader))
print("batch['image'].shape:", batch['image'].shape)
print("batch['agent_pos'].shape:", batch['agent_pos'].shape)
print("batch['action'].shape", batch['action'].shape)

batch['image'].shape:
torch.Size([64, 2, 3, 96, 96])

batch['agent_pos'].shape:
torch.Size([64, 2, 2])

batch['action'].shape
torch.Size([64, 16, 2])

# Models


## Vision Embedding Model


In [18]:

# options
# resnet18 efficientnet_b0 mobilenet_v2 mobilenet_v3_small mobilenet_v3_large
encoder = VisionEncoder()
vision_encoder, vision_feature_dim = encoder.get_model("resnet50")

## Unet Model


In [19]:
# Example Input: Define the observation and agent position tensors
image = torch.zeros((1, obs_horizon, 3, 96, 96))
agent_pos = torch.zeros((1, obs_horizon, 2))
lowdim_obs_dim = agent_pos.shape[-1] # agent_pos is 2 dimensional
# observation feature depends on the selected model output
obs_dim = vision_feature_dim + lowdim_obs_dim
action_dim = 2

In [20]:
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,  # input_dim (action_dim)
    global_cond_dim=obs_dim * obs_horizon,  # obs_dim=(512 + 2) * obs_horizon=2 -> 1028
)

nets = nn.ModuleDict({
    'vision_encoder': vision_encoder,
    'noise_pred_net': noise_pred_net
})

print(sum(p.numel() for p in nets.parameters()))

number of parameters 1.239875e+08


147495490

# Training


In [9]:
trainer = UnetTrainer(nets, dataloader)

model_path = os.path.dirname(os.getcwd()) + '/saved_models'
ema_nets = trainer.train_diffusion_policy(100, False, model_path)

wandb: Currently logged in as: tegbe to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch: 100%|██████████| 100/100 [1:31:07<00:00, 54.68s/it, loss=0.00671]


epoch/avg_loss,█▇▆▆▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
epoch/avg_timestep,▂▂▄▄▄▄▂▂▆▃▂▃▄█▆▆▃▂▂▄▁▄▆▆▂▁▆▆▃▃▅▅▆▃▄▃▆▅▆▅
epoch/final_lr,███████▇▇▇▇▇▇▇▆▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
epoch/max_loss,██▆▇▇▆▆▅▅▅▄▅▄▄▄▃▄▃▅▃▃▃▃▃▃▂▂▃▁▃▁▂▁▂▁▁▂▁▂▁
epoch/min_loss,█▃▃▃▃▃▃▂▂▂▂▁▂▂▂▂▁▂▂▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/number,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
epoch/std_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/batch_loss,█▆▅▅▅█▅▄▆▇▄▂▃▄▃▂▃▂▃▃▃▂▂▁▂▂▂▂▂▃▂▁▂▂▁▂▂▁▂▁
train/epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇█
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇█
train/grad_norm,█▃▄▄▄▂▂▁▂▃▂▃▂▁▂▂▂▃▄▁▃▄▂▅▁▂▃▃▃▃▂▄▄▁▄▄▁▁▄▃


Training completed and logged to wandb!


In [10]:
path = os.path.join(os.path.dirname(os.getcwd()), 'saved_models')
checkpoint_name = os.path.join(path, "MobileNetV3_unet_100_model_checkpoint.pth")
#torch.save(ema_nets.state_dict(), checkpoint_name)

In [11]:

# # load model
# # Load the state dictionary into the model
# load_ema_nets = nets
# load_ema_nets.load_state_dict(torch.load(checkpoint_name))

# # set to inference
# load_ema_nets.eval()

In [15]:
def run_inference(
    ema_nets,
    device,
    max_steps: int = 500,
    seed: int = 100000,
    save_video: bool = True,
    video_path: str = 'vis.mp4'
):
    """
    Run inference using the trained diffusion policy (no W&B logging).
    """
    # No wandb.init() here

    # Setup environment
    env.seed(seed)
    obs, info = env.reset()

    cfg = TrainingConfig()

    # Instead of setup_training(), manually build required components
    if device!='cpu':
        device = torch.device(cfg.device)
    noise_scheduler = DDPMScheduler(  # or whatever scheduler you use
        num_train_timesteps=cfg.num_diffusion_iters
    )

    obs_horizon = cfg.obs_horizon
    pred_horizon = cfg.pred_horizon
    action_horizon = cfg.action_horizon

    # Infer action_dim from env
    if hasattr(env.action_space, 'shape'):
        action_dim = env.action_space.shape[0]
    else:
        action_dim = 2  # fallback

    # Keep recent observations
    obs_deque = collections.deque([obs] * obs_horizon, maxlen=obs_horizon)

    imgs = [env.render(mode='rgb_array')] if save_video else []
    rewards = []
    done = False
    step_idx = 0

    with tqdm(total=max_steps, desc="Eval PushTImageEnv") as pbar:
        while not done:
            B = 1
            images = np.stack([x['image'] for x in obs_deque])
            agent_poses = np.stack([x['agent_pos'] for x in obs_deque])

            nagent_poses = normalize_data(agent_poses, stats=stats['agent_pos'])
            nimages = images

            nimages = torch.from_numpy(nimages).to(device, dtype=torch.float32)
            nagent_poses = torch.from_numpy(nagent_poses).to(device, dtype=torch.float32)

            with torch.no_grad():
                image_features = ema_nets['vision_encoder'](nimages)
                obs_features = torch.cat([image_features, nagent_poses], dim=-1)
                obs_cond = obs_features.unsqueeze(0).flatten(start_dim=1)

                noisy_action = torch.randn((B, pred_horizon, action_dim), device=device)
                naction = noisy_action

                noise_scheduler.set_timesteps(cfg.num_diffusion_iters)
                for k in noise_scheduler.timesteps:
                    noise_pred = ema_nets['noise_pred_net'](
                        sample=naction,
                        timestep=k,
                        global_cond=obs_cond
                    )
                    naction = noise_scheduler.step(
                        model_output=noise_pred,
                        timestep=k,
                        sample=naction
                    ).prev_sample

                naction = naction.detach().to('cpu').numpy()[0]
                action_pred = unnormalize_data(naction, stats=stats['action'])

                start = obs_horizon - 1
                end = start + action_horizon
                action = action_pred[start:end, :]

            for i in range(len(action)):
                obs, reward, done, _, info = env.step(action[i])
                obs_deque.append(obs)
                rewards.append(reward)
                if save_video:
                    imgs.append(env.render(mode='rgb_array'))
                step_idx += 1
                pbar.update(1)
                pbar.set_postfix(reward=reward)
                if step_idx > max_steps:
                    done = True
                    break

            if done:
                break

    score = max(rewards) if rewards else 0
    print(f'Score: {score}')

    if save_video and imgs:
        try:
            vwrite(video_path, imgs)
            Video(video_path, embed=True, width=256, height=256)
            print(f"Video saved to: {video_path}")
        except Exception as e:
            print(f"Error saving video: {e}")

run_inference(ema_nets,
              device=''
              )

Eval PushTImageEnv: 501it [00:41, 12.11it/s, reward=0]                         


Score: 0.0

Video saved to: vis.mp4

In [16]:
Video('vis.mp4', embed=True, width=256, height=256)

observations when threshold is 95% coverage

resnet

1. less than 5 epoch will not learn
2. less than 10 epoch will not learn
3. less than 20 epoch will underfit
4. less than 30 still underfiting
5. 40 is much better but still underfitting 133/200 [00:10<00:05, 13.08it/s, reward=1]
6. more reasonable results at 50, more work can be done 133/200 [00:10<00:05, 13.10it/s, reward=1]
7. 60 still slightly underfitting
8. 80 seems by far the most stable
9. 100 some unstable results but it works consistently

Efficient Net

1. 100 is just as poor as resnet 10 certain times it gets it almost right but doesnt know wen to stop

observations when threshold is 85% coverage

Mobile Net large

1. consistently got lost trying at 100, not reasonable at all.
